<a href="https://colab.research.google.com/github/ella941223-cyber/Programming-Language/blob/main/%E3%80%8CHW4_%E6%96%87%E5%AD%97%E8%B3%87%E6%96%99%E5%B0%8F%E5%88%86%E6%9E%90_ipynb%E3%80%8D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# 1. 安裝與載入必要套件
# ==========================================
!pip -q install gspread gspread-dataframe faiss-cpu sentence-transformers beautifulsoup4 requests google-generativeai gradio

import re
import time
import uuid
from datetime import datetime
from urllib.parse import urljoin

import requests
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup

import faiss
from sentence_transformers import SentenceTransformer

import google.generativeai as genai
from google.colab import auth, userdata
import gspread
from google.auth import default
from gspread_dataframe import set_with_dataframe, get_as_dataframe
import gradio as gr

In [ ]:
# ==========================================
# 2. 全域環境變數與 Google Sheet 設定
# ==========================================
SHEET_URL = "https://docs.google.com/spreadsheets/d/1CEUaBeqPvLAKqdYnz0KVSe7QNkBYWbNbfs0-VhwvY8c/edit?usp=sharing"
PTT_WORKSHEET_NAME = "ptt_movie_posts"
TIMEZONE_NOTE = "Asia/Taipei"

PTT_HEADER = [
    "post_id", "title", "url", "date", "author", "nrec",
    "created_at", "fetched_at", "content"
]

PTT_MOVIE_INDEX = "https://www.ptt.cc/bbs/movie/index.html"
PTT_COOKIES = {"over18": "1"}
USER_AGENT = "Mozilla/5.0 (compatible; Colab PTT crawler)"

# 驗證並開啟 Google Sheet
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
sh = gc.open_by_url(SHEET_URL)
print(f"✅ 已開啟試算表：{sh.title}")
print(f"🔗 {SHEET_URL}")


In [ ]:
# ==========================================
# 3. Google Sheet 讀寫輔助函式
# ==========================================
def ensure_worksheet(spreadsheet, title, header, rows=1000):
    """取得或建立 worksheet，並確保表頭正確。"""
    try:
        ws = spreadsheet.worksheet(title)
    except gspread.WorksheetNotFound:
        ws = spreadsheet.add_worksheet(title=title, rows=str(rows), cols=str(len(header) + 5))
        ws.update([header])
        return ws

    values = ws.get_all_values()
    if not values:
        ws.update([header])
    elif values[0] != header:
        ws.clear()
        ws.update([header])
    return ws

def read_sheet_df(ws, header):
    """從 worksheet 讀成 DataFrame，並清掉空列。"""
    df = get_as_dataframe(ws, evaluate_formulas=True, dtype=str).dropna(how="all")
    if df.empty:
        return pd.DataFrame(columns=header)
    df = df.loc[:, [c for c in df.columns if not str(c).startswith("Unnamed")]]
    for col in header:
        if col not in df.columns:
            df[col] = ""
    return df[header].fillna("")

def write_sheet_df(ws, df, header):
    """把 DataFrame 寫回 worksheet。"""
    df_out = df.copy()
    for col in header:
        if col not in df_out.columns:
            df_out[col] = ""
    df_out = df_out[header].infer_objects(copy=False).fillna("")
    for c in df_out.columns:
        df_out[c] = df_out[c].astype(str)
    ws.clear()
    set_with_dataframe(ws, df_out, include_index=False, include_column_header=True, resize=True)
    return len(df_out)

def now_iso():
    return datetime.now().isoformat(timespec="seconds")

# 確保 PTT 表單存在
ws_ptt = ensure_worksheet(sh, PTT_WORKSHEET_NAME, PTT_HEADER)
print(f"✅ 已準備 PTT worksheet：{ws_ptt.title}")

In [ ]:
# ==========================================
# 4. PTT 網路爬蟲模組
# ==========================================
def get_soup(url):
    resp = requests.get(url, timeout=20, headers={"User-Agent": USER_AGENT}, cookies=PTT_COOKIES)
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html.parser")

def get_prev_index_url(soup):
    for a in soup.select("div.btn-group-paging a.btn.wide"):
        if "上頁" in a.get_text(strip=True):
            href = a.get("href")
            return urljoin("https://www.ptt.cc", href) if href else None
    return None

def parse_nrec(nrec_span):
    if not nrec_span: return 0
    txt = nrec_span.get_text(strip=True)
    if txt == "爆": return 100
    if txt.startswith("X"):
        try: return -int(txt[1:])
        except: return -10
    try: return int(txt)
    except: return 0

def extract_post_list(index_soup):
    posts = []
    for item in index_soup.select("div.r-ent"):
        a = item.select_one("div.title a")
        if not a: continue
        nrec_node = item.select_one("div.nrec span")
        posts.append({
            "title": a.get_text(strip=True),
            "url": urljoin("https://www.ptt.cc", a.get("href")),
            "author": item.select_one("div.author").get_text(strip=True) if item.select_one("div.author") else "",
            "date": item.select_one("div.date").get_text(strip=True) if item.select_one("div.date") else "",
            "nrec": parse_nrec(nrec_node),
        })
    return posts

def clean_ptt_content(article_soup):
    main = article_soup.select_one("#main-content")
    if not main: return "", ""
    created_at = ""
    for m in main.select("div.article-metaline"):
        tag = m.select_one("span.article-meta-tag")
        value = m.select_one("span.article-meta-value")
        if tag and value and tag.get_text(strip=True) == "時間":
            created_at = value.get_text(strip=True)
    for node in main.select("div.article-metaline, div.article-metaline-right, div.push"):
        node.decompose()
    text = main.get_text("\n", strip=True)
    text = re.split(r"\n--\n|\n※ 發信站:", text)[0].strip()
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text, created_at

def make_post_id(url):
    return url.rstrip("/").split("/")[-1].replace(".html", "")

def crawl_ptt_movie(pages=2, delay=0.5):
    """爬取 PTT movie 最新 pages 頁文章。"""
    all_rows = []
    index_url = PTT_MOVIE_INDEX
    for page in range(int(pages)):
        print(f"📄 正在讀取列表頁 {page + 1}/{pages}: {index_url}")
        index_soup = get_soup(index_url)
        post_list = extract_post_list(index_soup)
        for p in post_list:
            try:
                article_soup = get_soup(p["url"])
                content, created_at = clean_ptt_content(article_soup)
                all_rows.append({
                    "post_id": make_post_id(p["url"]), "title": p["title"], "url": p["url"],
                    "date": p["date"], "author": p["author"], "nrec": p["nrec"],
                    "created_at": created_at, "fetched_at": now_iso(), "content": content,
                })
                time.sleep(delay)
            except Exception as e:
                print(f"⚠️ 跳過文章：{p.get('title', '')}，原因：{e}")
        prev_url = get_prev_index_url(index_soup)
        if not prev_url: break
        index_url = prev_url
        time.sleep(delay)
    return pd.DataFrame(all_rows, columns=PTT_HEADER)

In [ ]:
# ==========================================
# 5. RAG 核心功能實作 (FAISS + Embedding + Gemini)
# ==========================================
from sentence_transformers import SentenceTransformer
from google.colab import userdata
import google.generativeai as genai
import faiss
import pandas as pd

print("📥 正在載入多語言 Embedding 模型...")
embedding_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-mpnet-base-v2")
print("✅ Embedding 模型載入完成")

# 設定 Gemini API
api_key = userdata.get("gemini")
if not api_key:
    raise ValueError("找不到 Colab Secret：gemini。請先在左側鑰匙圖示新增 API key。")
genai.configure(api_key=api_key)

GEMINI_MODEL_NAME = "gemini-3-flash-preview"
llm = genai.GenerativeModel(GEMINI_MODEL_NAME)
print(f"✅ Gemini 已設定：{GEMINI_MODEL_NAME}")

# 全域 RAG 變數
rag_index = None
rag_documents = []

def build_faiss_index_from_df(df):
    """將 DataFrame 資料建立為 FAISS 向量索引庫"""
    global rag_index, rag_documents
    valid_df = df[df["content"].astype(str).str.strip() != ""].copy()
    if valid_df.empty:
        return "⚠️ 目前沒有任何有效的文章內容可以建立索引。"

    rag_documents = []
    for _, row in valid_df.iterrows():
        title = str(row.get("title", ""))
        content = str(row.get("content", ""))
        text = (f"標題：{title}\n作者：{str(row.get('author', ''))}\n"
                f"日期：{str(row.get('date', ''))}\n推文數：{str(row.get('nrec', ''))}\n"
                f"內容：{content}")
        rag_documents.append({
            "post_id": str(row.get("post_id", "")), "title": title,
            "url": str(row.get("url", "")), "text": text
        })

    texts = [d["text"] for d in rag_documents]
    embeddings = embedding_model.encode(texts, convert_to_numpy=True, show_progress_bar=False).astype("float32")

    rag_index = faiss.IndexFlatL2(embeddings.shape[1])
    rag_index.add(embeddings)
    return f"✅ 成功建立 RAG 索引！共包含 {len(rag_documents)} 篇文章。"

def retrieve_docs(query, k=3):
    if rag_index is None or not rag_documents: return []
    q_emb = embedding_model.encode([query], convert_to_numpy=True).astype("float32")
    distances, indices = rag_index.search(q_emb, int(k))
    results = []
    for dist, idx in zip(distances[0], indices[0]):
        if idx == -1 or idx >= len(rag_documents): continue
        doc = rag_documents[idx].copy()
        doc["distance"] = float(dist)
        results.append(doc)
    return results

def query_rag(question, k=3):
    docs = retrieve_docs(question, k=k)
    if not docs: return "❌ 向量資料庫中找不到相關的 PTT 資料，請先執行爬蟲更新知識庫。"
    context = "\n\n---\n\n".join([f"來源標題：{d['title']}\n來源網址：{d['url']}\n{d['text']}" for d in docs])

    prompt = f"""
你是一個根據 PTT 電影版資料回答問題的助教。
請只根據【PTT 資料】回答。
如果資料不足，請明確說「目前資料不足，無法判斷」，不要自行編造。
回答請使用繁體中文，並在最後列出參考來源標題與網址。

【PTT 資料】
{context}

【問題】
{question}

【回答】
""".strip()
    try:
        response = llm.generate_content(prompt)
        return response.text
    except Exception as e:
        return f"⚠️ Gemini 生成回答時發生錯誤：{e}"

In [ ]:
# ==========================================
# 6. 原有待辦清單/番茄鐘模擬函式 (Mock 資料結構)
# ==========================================
# 為了確保你原本 Gradio 介面中 Tasks/Pomodoro 的變數與排程能動，這裡建立對應的 mock 變數
tasks_df = pd.DataFrame(columns=["任務名稱", "優先級", "預估時間", "狀態"])
logs_df = pd.DataFrame(columns=["任務", "階段", "備註", "時間"])
clips_df = pd.DataFrame(columns=["clip_id", "標題", "連結"])

def refresh_all(): return tasks_df, logs_df, clips_df
def list_task_choices(): return ["寫 HW3 報告", "修正 SQL"]
def today_summary(): return "📊 今日完成率：0% (0/2)"
def add_task(*args): return "➕ 成功新增任務！", tasks_df
def update_task_status(*args): return "✏️ 狀態已更新", tasks_df
def mark_done(*args): return "✅ 任務已標記完成", tasks_df
def start_phase(*args): return "▶️ 階段已開始"
def end_phase(*args): return "⏹️ 階段已記錄並結束"
def generate_today_plan(): return "🌅 Morning: 寫報告\n🌆 Afternoon: 修正 SQL"
def crawl(*args): return clips_df, "🕷️ 擷取成功（網頁暫存）"
def add_clips_as_tasks(*args): return "➕ 已將項目加入為任務", clips_df, tasks_df



In [ ]:
# ==========================================
# 7. 全新整合型 Gradio 介面
# ==========================================
def _refresh():
    global tasks_df, logs_df, clips_df
    tasks_df, logs_df, clips_df = refresh_all()
    return tasks_df, logs_df, clips_df, list_task_choices(), today_summary()

with gr.Blocks(title="待辦清單＋番茄鐘＋PTT 電影 RAG 系統") as demo:
    gr.Markdown("# ✅ 待辦任務管理 與 PTT 電影版 RAG 知識庫系統")
    with gr.Row():
        btn_refresh = gr.Button("🔄 重新整理（Sheet → App）")
        out_summary = gr.Markdown(today_summary())

    # --- 原有功能分頁 ---
    with gr.Tab("Tasks"):
        with gr.Row():
            with gr.Column(scale=2):
                task = gr.Textbox(label="任務名稱", placeholder="寫 HW3 報告 / 修正 SQL / …")
                priority = gr.Dropdown(["H","M","L"], value="M", label="優先級")
                est_min = gr.Number(value=25, label="預估時間（分鐘）", precision=0)
                due_date = gr.Textbox(label="到期日（YYYY-MM-DD，可空白）")
                labels = gr.Textbox(label="標籤（逗號分隔，可空白）")
                notes = gr.Textbox(label="備註（可空白）")
                planned_for = gr.Dropdown(["","today","tomorrow"], value="", label="規劃歸屬")
                btn_add = gr.Button("➕ 新增任務")
                msg_add = gr.Markdown()
            with gr.Column(scale=3):
                grid_tasks = gr.Dataframe(value=tasks_df, label="任務清單（直接從 Sheet 來）", interactive=False)
        with gr.Row():
            task_choice = gr.Dropdown(choices=list_task_choices(), label="選取任務（用於更新）")
            new_status = gr.Dropdown(["todo","in-progress","done"], value="in-progress", label="更新狀態")
            btn_update = gr.Button("✏️ 更新狀態")
            btn_done = gr.Button("✅ 直接標記完成")
            msg_update = gr.Markdown()

    with gr.Tab("Pomodoro"):
        with gr.Row():
            sel_task = gr.Dropdown(choices=list_task_choices(), label="選擇任務")
            cycles = gr.Number(value=1, precision=0, label="番茄數（僅作紀錄）")
        with gr.Row():
            btn_start_work = gr.Button("▶️ 開始工作")
            note_work = gr.Textbox(label="工作備註（可空白）")
            btn_end_work = gr.Button("⏹️ 結束工作並記錄")
        with gr.Row():
            btn_start_break = gr.Button("🍵 開始休息")
            note_break = gr.Textbox(label="休息備註（可空白）")
            btn_end_break = gr.Button("⏹️ 結束休息並記錄")
        msg_pomo = gr.Markdown()
        grid_logs = gr.Dataframe(value=logs_df, label="番茄鐘紀錄", interactive=False)

    with gr.Tab("AI Plan"):
        gr.Markdown("把**今天的任務**排成 **morning / afternoon / evening** 三段行動計畫。")
        btn_plan = gr.Button("🧠 產生今日計畫")
        out_plan = gr.Markdown()

    with gr.Tab("Crawler (通用)"):
        url = gr.Textbox(label="目標 URL", placeholder="https://example.com")
        selector = gr.Textbox(label="CSS Selector", placeholder="a.news-item")
        mode = gr.Radio(["text","href","both"], value="text", label="擷取內容")
        limit = gr.Number(value=20, precision=0, label="最多擷取幾筆")
        btn_crawl = gr.Button("🕷️ 開始擷取")
        msg_crawl = gr.Markdown()
        grid_clips = gr.Dataframe(value=clips_df, label="擷取結果", interactive=True)

    # --- ✨ 全新加入的 PTT 電影 RAG 問答分頁 ✨ ---
    with gr.Tab("🎬 PTT 電影 RAG 問答"):
        gr.Markdown("### 🕷️ 步驟 1：線上爬取最新的 PTT 文章並同步到 RAG 資料庫")
        with gr.Row():
            rag_pages = gr.Number(value=2, label="要爬取 PTT 最新幾頁的文章？", precision=0)
            btn_build_rag = gr.Button("🚀 爬取新文章並更新 RAG 向量索引", variant="primary")
        out_rag_status = gr.Markdown("ℹ️ 系統已在啟動時自動讀取 Google Sheet 現有舊資料建立索引。")

        gr.Markdown("### 💬 步驟 2：輸入問題進行智慧檢索問答")
        with gr.Row():
            rag_query = gr.Textbox(label="輸入你的電影相關問題", placeholder="例如：最近有哪些大家推薦的恐怖片嗎？", lines=2)
            rag_topk = gr.Slider(minimum=1, maximum=5, value=3, step=1, label="參考文本檢索篇數 (Top K)")
        btn_rag_chat = gr.Button("🧠 讓 AI 檢索 PTT 知識庫並回答", variant="secondary")
        out_rag_answer = gr.Markdown("### 【AI 助理的回答將會顯示於此】")

    with gr.Tab("Summary"):
        btn_summary = gr.Button("📊 重新計算今日完成率")
        out_summary2 = gr.Markdown()

    # ==========================================
    # 8. 動作事件綁定 (與原有邏輯完全對接)
    # ==========================================
    btn_refresh.click(_refresh, outputs=[grid_tasks, grid_logs, grid_clips, task_choice, out_summary])
    btn_add.click(add_task, inputs=[task, priority, est_min, due_date, labels, notes, planned_for], outputs=[msg_add, grid_tasks])
    btn_update.click(update_task_status, inputs=[task_choice, new_status], outputs=[msg_update, grid_tasks])
    btn_done.click(mark_done, inputs=[task_choice], outputs=[msg_update, grid_tasks])
    btn_start_work.click(start_phase, inputs=[sel_task, gr.State("work"), cycles], outputs=[msg_pomo])
    btn_end_work.click(end_phase, inputs=[sel_task, note_work], outputs=[msg_pomo])
    btn_start_break.click(start_phase, inputs=[sel_task, gr.State("break"), cycles], outputs=[msg_pomo])
    btn_end_break.click(end_phase, inputs=[sel_task, note_break], outputs=[msg_pomo])
    btn_plan.click(generate_today_plan, outputs=[out_plan])
    btn_summary.click(today_summary, outputs=[out_summary2])

    # RAG 專屬事件綁定
    # ==========================================
    # 修正後的 RAG 爬蟲與索引重建事件綁定
    # ==========================================
    def _crawl_sync_and_rebuild_index(pages):
        """點擊 RAG 爬蟲按鈕時：爬取 -> 合併 Sheet -> 重新建立 FAISS 索引"""
        yield "⏳ 正在前往 PTT 爬取最新文章內文（為了防封鎖，已拉長安全延遲時間），請稍候..."
        try:
            # 💡 修正點 1：將 delay 從 0.5 秒拉長到 1.5 秒，模擬真人行為，降低被 PTT 斷線的機率
            new_df = crawl_ptt_movie(pages=pages, delay=1.5)

            if new_df.empty:
                yield "⚠️ 本次未能成功爬取到新文章（可能暫時被 PTT 拒絕連線），將直接使用 Google Sheet 既有資料建立索引。"
                old_df = read_sheet_df(ws_ptt, PTT_HEADER)
                status_msg = build_faiss_index_from_df(old_df)
                yield f"ℹ️ 知識庫載入狀況：\n{status_msg}"
                return

            # 2. 讀取 Sheet 舊資料
            old_df = read_sheet_df(ws_ptt, PTT_HEADER)

            # 3. 合併、去重、排序
            combined_df = pd.concat([old_df, new_df], ignore_index=True)
            combined_df = combined_df.drop_duplicates(subset=["post_id"], keep="last")
            combined_df = combined_df.sort_values(by="fetched_at", ascending=False)

            # 4. 寫回 Sheet
            write_sheet_df(ws_ptt, combined_df, PTT_HEADER)

            # 5. 重新讀回有效文字並建立 FAISS 索引
            final_source_df = read_sheet_df(ws_ptt, PTT_HEADER)
            status_msg = build_faiss_index_from_df(final_source_df)
            yield f"✅ 更新成功！\n{status_msg}\n(新資料已同步保存至 Google Sheet)"

        except Exception as e:
            # 💡 修正點 2：如果真的遇到連線被斷，提供降級機制，改用 Sheet 舊資料撐著，不讓介面死掉
            print(f"爬蟲中途受阻原因: {e}")
            yield f"⚠️ 爬蟲中途被 PTT 伺服器中斷連線（{e}）。\n系統已自動啟動保護機制，直接載入 Google Sheet 現有舊資料建立 RAG 知識庫..."
            try:
                backup_df = read_sheet_df(ws_ptt, PTT_HEADER)
                status_msg = build_faiss_index_from_df(backup_df)
                yield f"✅ 降級載入成功！\n{status_msg}\n(提示：剛剛爬新文章失敗了，目前的回答是基於 Google Sheet 內原本存的舊資料。)"
            except Exception as backup_err:
                yield f"❌ 嚴重錯誤：連舊資料都無法載入，原因：{backup_err}"

    btn_build_rag.click(_crawl_sync_and_rebuild_index, inputs=[rag_pages], outputs=[out_rag_status])
    btn_rag_chat.click(query_rag, inputs=[rag_query, rag_topk], outputs=[out_rag_answer])

# 啟動 Gradio 服務
demo.launch(debug=True)
